# Heated Rod — Lab Answers

Solver comparison and accuracy/speed tradeoff using Jacobi and Gauss–Seidel iteration on the implicit-Euler heat equation.

## Question 1 — Threshold time

**Question.** Run the animation until the event triggers. What is the estimated time $t$ (in seconds) when the sensor first reaches $T_{\text{target}}$? Report the value exactly as shown, and state which solver you used.

### Answer

The sensor first reaches the target temperature $T_{\text{target}} = 30^\circ\text{C}$ at

$$t^{*} = 8.113 \,\text{s}$$

using the **Gauss–Seidel solver**. The animation's overlay reports the event as `reached at t*=8.113s`.

## Question 2 — Solver comparison

**Question.** Run the animation with Jacobi and Gauss–Seidel. For each, report:
- average solve time per frame (ms),
- iterations near the end (or at the moment the event triggers),
- relative residual near the end.

Then answer: which iterative method is more efficient here, and why?

### Answer

**Comparison of Jacobi vs Gauss–Seidel** (same matrix, $A$ tridiagonal with $\alpha \approx 71.4$, so the diagonal-dominance ratio $(1+2\alpha)/(2\alpha) \approx 1.007$ is only marginal):

- **Jacobi**
  - average solve time per frame: $\approx 9.2 \,\text{ms}$
  - iterations near the event: $1000$ (hits `max_iter` every frame — does not reach the tolerance)
  - relative residual near the event: $\approx 3.1 \times 10^{-6}$ (above $\text{tol} = 10^{-6}$)
- **Gauss–Seidel**
  - average solve time per frame: $\approx 173 \,\text{ms}$
  - iterations near the event: $\approx 572$ (decreasing each frame as $x_0$ becomes a better guess)
  - relative residual near the event: $\approx 9.9 \times 10^{-7}$ (within tolerance)

**Which is more efficient and why?**

In terms of **iteration count and convergence quality, Gauss–Seidel is clearly more efficient**: it actually reaches the requested tolerance $10^{-6}$ in $\sim 570$–$745$ iterations, whereas Jacobi stalls at the $1000$-iteration cap without ever satisfying the stopping criterion. This is the classical theoretical result: for a symmetric, tridiagonal, diagonally-dominant matrix like this one, the spectral radii satisfy $\rho_{\text{GS}} = \rho_{\text{J}}^{\,2}$, so each Gauss–Seidel iteration is asymptotically worth two Jacobi iterations because it immediately reuses the just-updated components $x_j^{(k+1)}$ for $j < i$ instead of waiting for the next sweep.

The wall-clock numbers can mislead in the opposite direction: Jacobi looks "faster per frame" only because its update $x^{(k+1)} = D^{-1}(b - R x^{(k)})$ is one vectorised matrix–vector product, while my Gauss–Seidel uses a Python `for i in range(n)` loop. With a vectorised implementation (or in compiled code), Gauss–Seidel would also be faster in absolute time. In short, **Gauss–Seidel is the more efficient iterative method here** — it converges, while Jacobi essentially does not within the iteration budget.

## Question 3 — Accuracy vs speed tradeoff

**Question.** Pick one iterative solver (Jacobi or Gauss–Seidel) and run it twice with two tolerances (for example $\text{tol} = 10^{-6}$ and $\text{tol} = 10^{-10}$). Compare:
- iteration counts,
- average solve time per frame,
- the reported $t^{*}$.

Does it change meaningfully? Explain the tradeoff you observe between speed and accuracy in this setup.

### Answer

**Solver:** Gauss–Seidel, run twice with $\text{tol} = 10^{-6}$ and $\text{tol} = 10^{-10}$.

- **$\text{tol} = 10^{-6}$**
  - iterations near the event: $\approx 572$
  - average solve time per frame: $\approx 175.5\,\text{ms}$
  - reported event time: $t^{*} = 8.1136\,\text{s}$
  - relative residual at the last frame: $\approx 9.94 \times 10^{-7}$
- **$\text{tol} = 10^{-10}$**
  - iterations near the event: $\approx 1147$
  - average solve time per frame: $\approx 336.7\,\text{ms}$
  - reported event time: $t^{*} = 8.1133\,\text{s}$
  - relative residual at the last frame: $\approx 9.89 \times 10^{-11}$

**Does it change meaningfully?**

The reported event time barely changes: $\Delta t^{*} \approx 3 \times 10^{-4}\,\text{s}$, which is roughly $0.06\%$ of the time step $\Delta t = 0.5\,\text{s}$ and far below the resolution at which $t^{*}$ is displayed ($3$ decimal places). In contrast, the **cost roughly doubles**: the iteration count goes from $\sim 572$ to $\sim 1147$ and the average solve time per frame goes from $\approx 175\,\text{ms}$ to $\approx 337\,\text{ms}$, while the relative residual itself drops by $4$ orders of magnitude exactly as requested.

**Tradeoff observed.**

This is a clear illustration that beyond a certain point, **tightening the linear-solver tolerance buys precision the rest of the simulation cannot use**. The dominant errors here are the time-stepping error ($O(\Delta t)$ for implicit Euler) and the linear interpolation used to estimate $t^{*}$ between two frames; both are far larger than $10^{-6}$. Driving $\text{tol}$ from $10^{-6}$ to $10^{-10}$ therefore wastes work to refine the residual of $A x = b$ in a regime where the physical answer no longer changes. In practice, the right choice is to set $\text{tol}$ slightly below the discretisation error of the outer scheme: small enough that the linear solve is not the bottleneck of accuracy, but not so small that Gauss–Seidel keeps iterating purely to chase round-off.